# Experiment 5: Variational Autoencoder (VAE) on MNIST

Runtime: **Runtime > Change runtime type > GPU**, then run the cells top to bottom.
At the end, download `vae_models.zip` and use it in the Flask app.

## Steps 1-2: Setup and imports

In [ ]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Steps 3-5: Load MNIST, normalize to 0-1, add channel dimension

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train = np.expand_dims(x_train, -1)   # (60000, 28, 28, 1)
x_test = np.expand_dims(x_test, -1)     # (10000, 28, 28, 1)

print("Train:", x_train.shape, "| Test:", x_test.shape)

## Step 6: Display sample images

In [ ]:
plt.figure(figsize=(12, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(int(y_train[i]))
    plt.axis("off")
plt.suptitle("Sample MNIST images", y=1.05)
plt.show()

## Step 7: VAE configuration

In [ ]:
ORIGINAL_DIM = 784    # 28 x 28
LATENT_DIM = 2        # 2-D latent space (z1, z2)
BATCH_SIZE = 128
EPOCHS = 30

## Steps 8-10: Encoder (z_mean, z_log_var) and Reparameterization Trick

In [ ]:
@keras.saving.register_keras_serializable(package="vae")
class Sampling(layers.Layer):
    """Reparameterization trick: z = mu + sigma * epsilon, epsilon ~ N(0, I)."""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        sigma = tf.exp(0.5 * z_log_var)          # sigma = exp(0.5 * log(sigma^2))
        return z_mean + sigma * epsilon


encoder_inputs = keras.Input(shape=(28, 28, 1), name="encoder_input")
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dense(256, activation="relu")(x)
z_mean = layers.Dense(LATENT_DIM, name="z_mean")(x)
z_log_var = layers.Dense(LATENT_DIM, name="z_log_var")(x)
z = Sampling(name="z")([z_mean, z_log_var])

encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

## Step 11: Decoder (Dense layers + Reshape)

In [ ]:
latent_inputs = keras.Input(shape=(LATENT_DIM,), name="decoder_input")
x = layers.Dense(256, activation="relu")(latent_inputs)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dense(ORIGINAL_DIM, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28, 1))(x)

decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

## Steps 12-13: VAE model with custom `train_step()` (Reconstruction Loss + KL Divergence)

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [self.total_loss_tracker,
                self.reconstruction_loss_tracker,
                self.kl_loss_tracker]

    def train_step(self, data):
        if isinstance(data, tuple):
            data = data[0]

        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Reconstruction loss: binary cross-entropy summed over the 784 pixels
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction),
                    axis=(1, 2),
                )
            )

            # KL divergence between N(mu, sigma^2) and N(0, I)
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))

            total_loss = reconstruction_loss + kl_loss   # L_VAE = L_recon + L_KL

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }


vae = VAE(encoder, decoder, name="vae")
vae.compile(optimizer=keras.optimizers.Adam())

## Step 14: Train

In [ ]:
history = vae.fit(x_train, epochs=EPOCHS, batch_size=BATCH_SIZE)

## Step 15: Plot total, reconstruction and KL loss

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(
    axes,
    ["loss", "reconstruction_loss", "kl_loss"],
    ["Total VAE loss", "Reconstruction loss", "KL loss"],
):
    ax.plot(range(1, EPOCHS + 1), history.history[key], marker="o", markersize=3)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 16: Original vs reconstructed test images

In [ ]:
n = 10
z_mean_test, _, _ = encoder.predict(x_test[:n], verbose=0)
reconstructed = decoder.predict(z_mean_test, verbose=0)

plt.figure(figsize=(15, 3.5))
for i in range(n):
    plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("Original", loc="left")

    plt.subplot(2, n, n + i + 1)
    plt.imshow(reconstructed[i].squeeze(), cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.title("Reconstructed", loc="left")
plt.show()

## Step 17: Visualize the 2-D latent space (z_mean)

In [ ]:
z_mean_all, _, _ = encoder.predict(x_test, batch_size=1024, verbose=0)

plt.figure(figsize=(8, 7))
sc = plt.scatter(z_mean_all[:, 0], z_mean_all[:, 1], c=y_test, cmap="tab10", s=3, alpha=0.7)
plt.colorbar(sc, ticks=range(10), label="Digit")
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("2-D latent space of MNIST test images")
plt.show()

Extra: a grid of digits decoded across the latent space. The Flask app uses the same idea as its clickable map.

In [ ]:
steps, limit = 15, 3.0
xs = np.linspace(-limit, limit, steps)
ys = np.linspace(limit, -limit, steps)

canvas = np.zeros((steps * 28, steps * 28))
for i, yv in enumerate(ys):
    for j, xv in enumerate(xs):
        digit = decoder.predict(np.array([[xv, yv]], dtype="float32"), verbose=0)[0, :, :, 0]
        canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = digit

plt.figure(figsize=(8, 8))
plt.imshow(canvas, cmap="gray", extent=[-limit, limit, -limit, limit])
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Digits decoded across the latent space")
plt.show()

## Step 18: Generate new images from random latent vectors, N(0, I)

In [ ]:
rng = np.random.default_rng()
z_random = rng.standard_normal((16, LATENT_DIM)).astype("float32")
generated = decoder.predict(z_random, verbose=0)

plt.figure(figsize=(7, 7))
for i in range(16):
    plt.subplot(4, 4, i + 1)
    plt.imshow(generated[i].squeeze(), cmap="gray")
    plt.axis("off")
plt.suptitle("New digits generated by the Decoder")
plt.show()

## Step 19: Generate a single new image

In [ ]:
z_single = np.random.normal(size=(1, LATENT_DIM)).astype("float32")
single = decoder.predict(z_single, verbose=0)[0, :, :, 0]

plt.imshow(single, cmap="gray")
plt.title(f"z = ({z_single[0, 0]:.2f}, {z_single[0, 1]:.2f})")
plt.axis("off")
plt.show()

## Step 20: Save Encoder and Decoder (Keras format)

In [ ]:
os.makedirs("vae_models", exist_ok=True)
encoder.save("vae_models/vae_encoder.keras")
decoder.save("vae_models/vae_decoder.keras")
print(os.listdir("vae_models"))

## Step 21: Load the saved Decoder and verify

In [ ]:
loaded_decoder = keras.models.load_model("vae_models/vae_decoder.keras", compile=False)

z_test = np.random.normal(size=(1, LATENT_DIM)).astype("float32")
new_image = loaded_decoder.predict(z_test, verbose=0)[0, :, :, 0]

plt.imshow(new_image, cmap="gray")
plt.title("Generated by the loaded Decoder")
plt.axis("off")
plt.show()

## Step 22: ZIP the models and download

Downloading the ZIP avoids Google Drive mounting. Put the extracted `vae_decoder.keras` in the Flask app's `models/` folder.

In [ ]:
shutil.make_archive("vae_models", "zip", "vae_models")

from google.colab import files
files.download("vae_models.zip")

# Use the same TensorFlow version locally to avoid load errors
print("Install locally with: pip install tensorflow==" + tf.__version__)

## Conclusion

The VAE was trained on MNIST using Reconstruction Loss and KL Divergence. The Encoder maps each image to a 2-D Gaussian (`z_mean`, `z_log_var`), the Reparameterization Trick samples `z`, and the Decoder turns `z` back into a 28x28 image. Sampling `z` from N(0, I) and passing it to the Decoder generates new MNIST-like digits.